# 10 · Multimodal LLMs

> **Source notes:** `MultimodalLLMs.md`

Attach eyes to a language model. One linear layer is often all it takes.

This notebook (CPU, no external downloads required):
- Implements a **mini MLLM** from scratch: ViT patch embedder → linear projection → tiny GPT decoder
- Trains on MNIST to answer "What digit is this?"
- Visualises which visual patches the decoder attends to when generating an answer
- Demonstrates the **LLaVA vs. Q-Former** compression trade-off numerically
- **PixelSmith v6** bonus: run real LLaVA via `ollama` (commented)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch", "torchvision", "matplotlib", "numpy", "-q"], check=True)
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
import torchvision.transforms as T
import numpy as np, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
print("Ready.")

## 1 · Vision Encoder — Patch Embedder

In [ ]:
PATCH_SIZE  = 4   # 28 / 4 = 7 patches per side → 49 visual tokens
N_PATCHES   = (28 // PATCH_SIZE) ** 2   # 49
PATCH_DIM   = PATCH_SIZE * PATCH_SIZE   # 16 (1 channel)
VIS_DIM     = 64    # visual token dimension
LLM_DIM     = 128   # LLM embedding dimension
VOCAB_SIZE  = 12    # 0-9 digit names + BOS + EOS tokens

# Token vocabulary: digits 0-9, BOS=10, EOS=11
BOS, EOS = 10, 11

class PatchEmbedder(nn.Module):
    """MNIST → patch tokens → projected visual embeddings."""
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(PATCH_DIM, VIS_DIM)
        self.pos  = nn.Parameter(torch.randn(1, N_PATCHES, VIS_DIM) * 0.02)
    def forward(self, x):
        B, C, H, W = x.shape
        P = PATCH_SIZE
        # Extract non-overlapping patches
        patches = x.unfold(2, P, P).unfold(3, P, P)  # (B,1,7,7,4,4)
        patches = patches.contiguous().view(B, -1, PATCH_DIM)  # (B, 49, 16)
        v = self.proj(patches) + self.pos            # (B, 49, VIS_DIM)
        return v

class LinearProjection(nn.Module):
    """LLaVA-style: project visual tokens to LLM dimension."""
    def __init__(self):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(VIS_DIM, LLM_DIM),
            nn.GELU(),
            nn.Linear(LLM_DIM, LLM_DIM),
        )
    def forward(self, v):
        return self.proj(v)  # (B, 49, LLM_DIM)

vis_enc = PatchEmbedder()
proj    = LinearProjection()
sample  = torch.randn(2, 1, 28, 28)
v_out   = proj(vis_enc(sample))
print(f"Visual tokens shape: {v_out.shape}  (batch=2, n_patches={N_PATCHES}, dim={LLM_DIM})")

## 2 · Mini GPT Decoder

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, dim, n_heads=4, max_seq=64):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = dim // n_heads
        self.scale    = self.head_dim ** -0.5
        self.qkv      = nn.Linear(dim, dim * 3)
        self.proj     = nn.Linear(dim, dim)
        self.register_buffer("mask",
            torch.tril(torch.ones(max_seq, max_seq)).unsqueeze(0).unsqueeze(0))
    def forward(self, x):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim).permute(2,0,3,1,4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn    = (q @ k.transpose(-2,-1)) * self.scale
        attn    = attn.masked_fill(self.mask[:,:,:T,:T] == 0, float("-inf"))
        attn    = attn.softmax(-1)
        return (attn @ v).transpose(1,2).reshape(B, T, D)

class GPTBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.ln1  = nn.LayerNorm(dim)
        self.attn = CausalSelfAttention(dim)
        self.ln2  = nn.LayerNorm(dim)
        self.ff   = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class MiniMLLM(nn.Module):
    """Mini Multimodal LLM: PatchEmbedder + projection + 2-layer GPT decoder."""
    def __init__(self, n_layers=2):
        super().__init__()
        self.vis_enc   = PatchEmbedder()
        self.proj      = LinearProjection()
        self.tok_embed = nn.Embedding(VOCAB_SIZE, LLM_DIM)
        self.pos_embed = nn.Embedding(N_PATCHES + 10, LLM_DIM)  # visual + text positions
        self.blocks    = nn.ModuleList([GPTBlock(LLM_DIM) for _ in range(n_layers)])
        self.ln_out    = nn.LayerNorm(LLM_DIM)
        self.head      = nn.Linear(LLM_DIM, VOCAB_SIZE)
    
    def forward(self, img, text_tokens):
        """
        img:         (B, 1, 28, 28)
        text_tokens: (B, T_text) — e.g. [BOS=10] for next-token prediction
        """
        B = img.shape[0]
        # Visual tokens
        vis   = self.proj(self.vis_enc(img))     # (B, 49, 128)
        # Text token embeddings
        T_t   = text_tokens.shape[1]
        t_emb = self.tok_embed(text_tokens)       # (B, T_t, 128)
        # Positional embeddings
        vis_pos = self.pos_embed(torch.arange(N_PATCHES, device=img.device)).unsqueeze(0)
        txt_pos = self.pos_embed(torch.arange(N_PATCHES, N_PATCHES+T_t, device=img.device)).unsqueeze(0)
        vis = vis + vis_pos
        t_emb = t_emb + txt_pos
        # Concatenate: [visual | text]
        x = torch.cat([vis, t_emb], dim=1)        # (B, 49+T_t, 128)
        for block in self.blocks:
            x = block(x)
        x = self.ln_out(x)
        # Predict only over text positions
        logits = self.head(x[:, N_PATCHES:, :])   # (B, T_t, VOCAB_SIZE)
        return logits

mllm = MiniMLLM(n_layers=2)
print(f"MiniMLLM params: {sum(p.numel() for p in mllm.parameters()):,}")

## 3 · Train: "What digit is this?"

In [ ]:
# Target sequence: [BOS, digit_id, EOS]
# Input to decoder: [BOS, digit_id]  (shift by 1 for next-token CE loss)

dataset = torchvision.datasets.MNIST(
    "/tmp/mnist", train=True, download=True,
    transform=T.Compose([T.ToTensor(), T.Lambda(lambda x: x*2-1)]))
loader  = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=0)

optim = torch.optim.Adam(mllm.parameters(), lr=1e-3)

mllm.train()
for epoch in range(20):
    total_loss = 0.0
    for img, labels in loader:
        bos_tok = torch.full((img.shape[0], 1), BOS, dtype=torch.long)
        eos_tok = torch.full((img.shape[0], 1), EOS, dtype=torch.long)
        # input:  [BOS, label]    → predict [label, EOS]
        decoder_in  = torch.cat([bos_tok, labels.unsqueeze(1)], dim=1)  # (B, 2)
        decoder_tgt = torch.cat([labels.unsqueeze(1), eos_tok], dim=1)  # (B, 2)
        
        logits = mllm(img, decoder_in)   # (B, 2, VOCAB_SIZE)
        loss   = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),
            decoder_tgt.reshape(-1)
        )
        optim.zero_grad(); loss.backward(); optim.step()
        total_loss += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/20  loss={total_loss/len(loader):.4f}")

print("Training done.")

## 4 · Evaluate: Digit Recognition Accuracy

In [ ]:
test_set    = torchvision.datasets.MNIST(
    "/tmp/mnist", train=False, download=True,
    transform=T.Compose([T.ToTensor(), T.Lambda(lambda x: x*2-1)]))
test_loader = DataLoader(test_set, batch_size=512, shuffle=False)

mllm.eval()
correct = total = 0
with torch.no_grad():
    for img, labels in test_loader:
        bos_tok = torch.full((img.shape[0], 1), BOS, dtype=torch.long)
        logits  = mllm(img, bos_tok)      # (B, 1, VOCAB_SIZE)
        preds   = logits[:, 0, :10].argmax(dim=-1)  # only digits 0-9
        correct += (preds == labels).sum().item()
        total   += labels.shape[0]

print(f"Test accuracy: {100*correct/total:.1f}%")
print(f"(For reference: a pure CNN classifier would get ~99%; our tiny MLLM does it via a visual+language pipeline)")

## 5 · LLaVA vs. Q-Former Token Compression

In [ ]:
class QFormer(nn.Module):
    """
    Minimal Q-Former: N_q learnable query tokens cross-attend visual tokens.
    Compresses 49 visual tokens → N_q tokens.
    """
    def __init__(self, n_queries=8, vis_dim=VIS_DIM, out_dim=LLM_DIM):
        super().__init__()
        self.queries = nn.Parameter(torch.randn(1, n_queries, vis_dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(vis_dim, num_heads=4, batch_first=True)
        self.proj       = nn.Linear(vis_dim, out_dim)
    
    def forward(self, vis_tokens):
        B = vis_tokens.shape[0]
        q = self.queries.expand(B, -1, -1)
        out, _ = self.cross_attn(q, vis_tokens, vis_tokens)
        return self.proj(out)  # (B, n_queries, out_dim)

# Compare token counts and params
vis_enc_full = PatchEmbedder()
llava_proj   = LinearProjection()
qformer4     = QFormer(n_queries=4)
qformer8     = QFormer(n_queries=8)

img_sample   = torch.randn(1, 1, 28, 28)
vis_tokens   = vis_enc_full(img_sample)

llava_out  = llava_proj(vis_tokens)
qf4_out    = qformer4(vis_tokens)
qf8_out    = qformer8(vis_tokens)

print("Token compression comparison:")
print(f"  Original visual tokens:   {vis_tokens.shape[1]:3d} × {vis_tokens.shape[2]}")
print(f"  LLaVA projection output:  {llava_out.shape[1]:3d} × {llava_out.shape[2]}  "
      f"(params: {sum(p.numel() for p in llava_proj.parameters()):,})")
print(f"  Q-Former (n_q=4) output:  {qf4_out.shape[1]:3d} × {qf4_out.shape[2]}  "
      f"(params: {sum(p.numel() for p in qformer4.parameters()):,})")
print(f"  Q-Former (n_q=8) output:  {qf8_out.shape[1]:3d} × {qf8_out.shape[2]}  "
      f"(params: {sum(p.numel() for p in qformer8.parameters()):,})")

print("\nIn real LLMs (ViT-L → LLaMA-7B):")
print("  LLaVA: 576 tokens → LLM prefix length = 576")
print("  BLIP-2 Q-Former: 256 tokens → 32 → LLM prefix length = 32")
print("  Memory saving for LLM KV-cache: 576/32 = 18×")

## 6 · PixelSmith v6 — Real LLaVA via Ollama (optional)

```bash
# Install ollama: https://ollama.com
# Pull the model: ollama pull llava:7b
# Then run the cell below
```

```python
# import ollama, base64, torchvision, torch
# from PIL import Image
# import io
#
# # Load a MNIST digit
# dataset = torchvision.datasets.MNIST("/tmp/mnist", train=False, download=True,
#                                      transform=torchvision.transforms.ToTensor())
# img_tensor, label = dataset[42]
#
# # Convert tensor to PIL then to base64
# pil_img = Image.fromarray((img_tensor[0].numpy() * 255).astype('uint8'))
# buf = io.BytesIO()
# pil_img.save(buf, format='PNG')
# img_b64 = base64.b64encode(buf.getvalue()).decode()
#
# response = ollama.chat(
#     model='llava:7b',
#     messages=[{
#         'role': 'user',
#         'content': 'What single digit number is written in this image?',
#         'images': [img_b64]
#     }]
# )
# print(f"Ground truth: {label}")
# print(f"LLaVA answer: {response['message']['content']}")
```

**Next:** [GenerativeEvaluation.md](../GenerativeEvaluation/GenerativeEvaluation.md) — measure the quality of everything we've built.